## Import Libraries

In [101]:
#Data Handling 
import pandas as pd
import numpy as np

#Json
import json

#SQL
import sqlite3

#API
import requests

# Profling
import sweetviz as sv

# Imputation
from sklearn.impute import SimpleImputer

# KNN
from sklearn.impute import KNNImputer

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

from scipy.stats import zscore

from scipy.stats.mstats import winsorize

from sklearn.preprocessing import OrdinalEncoder

from sklearn.preprocessing import LabelEncoder

from sklearn.preprocessing import StandardScaler

## PART B — Data Acquisition

**Load CSV File**

In [102]:
df = pd.read_csv('transactions.csv')
print(df.head())

  transaction_id customer_id transaction_date transaction_type  \
0         T00001       C0103       2025-07-08          Payment   
1         T00002       C0436       2025-03-26          Payment   
2         T00003       C0349       2022-02-16         Transfer   
3         T00004       C0271       2022-11-20         Transfer   
4         T00005       C0107       2024-04-23         Transfer   

   transaction_amount payment_method merchant_category  
0             7625.32    Net Banking             Other  
1             9527.03           Cash        Healthcare  
2             1243.62           Card        Healthcare  
3             2444.64    Net Banking            Travel  
4             2432.12           Card             Other  


**Load JSON File**

In [103]:

with open("customer_metadata.json", "r") as file:
    customer_data = json.load(file)

customer_df = pd.DataFrame(customer_data)

print("Customer Metadata")
print(customer_df.head())

Customer Metadata
  customer_id  age  gender       city     occupation education_level  \
0       C0001   34  Female     Jaipur        Private        Graduate   
1       C0002   47  Female  Hyderabad  Self-Employed        Graduate   
2       C0003   54  Female      Surat     Government        Graduate   
3       C0004   64  Female  Ahmedabad     Government   Post-Graduate   
4       C0005   43    Male     Mumbai     Government        Graduate   

   annual_income  
0      471001.85  
1      582371.99  
2      526826.66  
3      490360.69  
4      959929.91  


**Load SQL File**

In [104]:
conn = sqlite3.connect("loan_repayment.db")

loan_df = pd.read_sql(
    "SELECT * FROM loan_repayment",
    conn
)

print("Loan Repayment History")
print(loan_df.head())

Loan Repayment History
  loan_id customer_id  loan_amount  emi_amount repayment_status  days_late  \
0  L00001       C0438    211742.46    49422.55          On-Time         15   
1  L00002       C0170    826662.22    19308.19          On-Time         60   
2  L00003       C0375    211877.58    28938.01          On-Time          0   
3  L00004       C0045    371206.34    28008.41          On-Time          0   
4  L00005       C0381    201056.37    47948.07          On-Time          0   

        repayment_date  
0  2025-09-12 00:00:00  
1  2023-01-09 00:00:00  
2  2023-12-16 00:00:00  
3  2023-03-25 00:00:00  
4  2024-06-16 00:00:00  


In [105]:
merged_df = pd.merge(
    df,
    customer_df,
    on="customer_id",
    how="left"
)

print("Transaction + Customer:")
print(merged_df.head())


df = pd.merge(
    merged_df,
    loan_df,
    on="customer_id",
    how="left"
)

print("\nAfter Loan Repayment Merge:")
print(df.head())


# ==========================================
# API / Economic JSON Data
# ==========================================

import json

with open("economic_indicators_api_data.json", "r") as file:
    economic_data = json.load(file)

economic_df = pd.DataFrame(economic_data["data"])

print("\nEconomic Data:")
print(economic_df.head())


# ==========================================
# Create Year from Transaction Date
# ==========================================

df["transaction_date"] = pd.to_datetime(
    df["transaction_date"],
    errors="coerce"
)

df["year"] = df["transaction_date"].dt.year


# ==========================================
# Keep India Economic Data
# ==========================================

india_economic_df = economic_df[
    economic_df["country"] == "India"
].copy()

india_economic_df = india_economic_df.drop(
    columns=["country"]
)


# ==========================================
# Final Merge
# ==========================================

df = pd.merge(
    df,
    india_economic_df,
    on="year",
    how="left"
)

print("\n========== FINAL MERGED DATASET ==========")
print(merged_df.head())

print("\nFinal Shape:")
print(merged_df.shape)

Transaction + Customer:
  transaction_id customer_id transaction_date transaction_type  \
0         T00001       C0103       2025-07-08          Payment   
1         T00002       C0436       2025-03-26          Payment   
2         T00003       C0349       2022-02-16         Transfer   
3         T00004       C0271       2022-11-20         Transfer   
4         T00005       C0107       2024-04-23         Transfer   

   transaction_amount payment_method merchant_category  age  gender  \
0             7625.32    Net Banking             Other   59    Male   
1             9527.03           Cash        Healthcare   46    Male   
2             1243.62           Card        Healthcare   47  Female   
3             2444.64    Net Banking            Travel   60  Female   
4             2432.12           Card             Other   58    Male   

        city     occupation education_level  annual_income  
0      Delhi     Freelancer   Post-Graduate      746575.63  
1     Mumbai       Business   

## Q4 — Explore Dataset using Pandas .info() and .describe()

In [106]:
# First Five Row Of Dataset
print("=========================== First Five Row ================================")
print(df.head())

# Dataset information
print("=========================== Dataset Information ===========================")
print(df.info())

# NUMERICAL SUMMARY
print("========================== NUMERICAL SUMMARY ==============================")
print(df.describe())

# MISSING VALUES
print("========================= MISSING VALUES ==================================")
print(df.isnull().sum())

=========================== First Five Row ================================
  transaction_id customer_id transaction_date transaction_type  \
0         T00001       C0103       2025-07-08          Payment   
1         T00002       C0436       2025-03-26          Payment   
2         T00003       C0349       2022-02-16         Transfer   
3         T00004       C0271       2022-11-20         Transfer   
4         T00004       C0271       2022-11-20         Transfer   

   transaction_amount payment_method merchant_category  age  gender    city  \
0             7625.32    Net Banking             Other   59    Male   Delhi   
1             9527.03           Cash        Healthcare   46    Male  Mumbai   
2             1243.62           Card        Healthcare   47  Female  Mumbai   
3             2444.64    Net Banking            Travel   60  Female  Mumbai   
4             2444.64    Net Banking            Travel   60  Female  Mumbai   

   ... loan_id loan_amount  emi_amount repayment_sta

## Q5 - Data Profiling using Sweetviz

In [107]:
report = sv.analyze(df)

report.show_html(
    "data_profling_report.html",
    open_browser=True
)

print("Data Profling Report generated successfully!")

                                             |          | [  0%]   00:00 -> (? left)

Report data_profling_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.
Data Profling Report generated successfully!


## Q6 — Handle Missing Data

**Q6(a) Simple Imputer — Numerical: Mean/Median**

In [108]:
# Find Missing Value
print("=================================== Missing Value Before Imputation ===================================")
print(df.isnull().sum())

# Identify Numercial Columns

numercial_columns = df.select_dtypes(
    include=['int64','float64']
).columns

print("Numercial Columns:")
print(numercial_columns.tolist())

=================================== Missing Value Before Imputation ===================================
transaction_id          0
customer_id             0
transaction_date        0
transaction_type        0
transaction_amount      0
payment_method          0
merchant_category       0
age                     0
gender                  0
city                    0
occupation              0
education_level         0
annual_income           0
loan_id               384
loan_amount           384
emi_amount            384
repayment_status      384
days_late             384
repayment_date        384
year                    0
inflation               0
gdp_growth              0
unemployment            0
dtype: int64
Numercial Columns:
['transaction_amount', 'age', 'annual_income', 'loan_amount', 'emi_amount', 'days_late', 'inflation', 'gdp_growth', 'unemployment']


**Median Imputation**

In [109]:
df_median = df.copy()

imputer = SimpleImputer(strategy="median")

df_median[numercial_columns] = imputer.fit_transform(
    df_median[numercial_columns]
)

print("============================ Missing Value After Median Imputation ==================================")
print(df_median[numercial_columns].isnull().sum())

============================ Missing Value After Median Imputation ==================================
transaction_amount    0
age                   0
annual_income         0
loan_amount           0
emi_amount            0
days_late             0
inflation             0
gdp_growth            0
unemployment          0
dtype: int64


**Q6(b) Simple Imputer — Categorical: Most Frequent**

In [110]:
categorical_columns = df.select_dtypes(
    include=['object']
).columns

print("============================== Categorical Columns ==========================================")
print(categorical_columns.tolist())


============================== Categorical Columns ==========================================
['transaction_id', 'customer_id', 'transaction_type', 'payment_method', 'merchant_category', 'gender', 'city', 'occupation', 'education_level', 'loan_id', 'repayment_status', 'repayment_date']


C:\Users\Dell\AppData\Local\Temp\ipykernel_28564\1589591333.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(


**Now imputation:**

In [111]:
df_categorical = df.copy()

cat_imputer = SimpleImputer(
    strategy="most_frequent"
)

df_categorical[categorical_columns] = cat_imputer.fit_transform(
    df_categorical[categorical_columns]
)

print("Missing Value After Categorical Imputation:")
print(df_categorical[categorical_columns].isnull().sum())

Missing Value After Categorical Imputation:
transaction_id       0
customer_id          0
transaction_type     0
payment_method       0
merchant_category    0
gender               0
city                 0
occupation           0
education_level      0
loan_id              0
repayment_status     0
repayment_date       0
dtype: int64


**Q6(c) Most Frequent Category Imputation**

In [112]:
df_mode = df.copy()

column = "gender"

most_frequent = df_mode[column].mode()[0]

df_mode[column] = df_mode[column].fillna(
    most_frequent
) 

print("Most Frequent Category:", most_frequent)

print("Missing Value After Imputation:",
      df_mode[column].isnull().sum()
      )

Most Frequent Category: Male
Missing Value After Imputation: 0


**Q6(d) Missing Indicator + Random Sample Imputation**

In [113]:
df_random = df.copy()

#Missing Indicator 
df_random["annual_income_missing"] = (
    df_random["annual_income"].isnull().astype(int)
)

print("Missing Indicator:")
print(
    df_random["annual_income_missing"].value_counts()
)

Missing Indicator:
annual_income_missing
0    2470
Name: count, dtype: int64


**Sample imputation:**

In [114]:
missing_index = df_random[
    "annual_income"
].isnull()

available_values = df_random[
    "annual_income"
].dropna()

random_values = np.random.choice(
    available_values,
    size=missing_index.sum(),
    replace=True
)

df_random.loc[
    missing_index,
    "annual_income"
] = random_values

print(
    "Missing annual_income after imputation:",
    df_random["annual_income"].isnull().sum()
)

Missing annual_income after imputation: 0


**Q6(e) KNN Imputer — Multivariate**

In [115]:

df_knn = merged_df.copy()

knn_columns = df_knn.select_dtypes(
    include=["int64", "float64"]
).columns

knn_imputer = KNNImputer(
    n_neighbors=5
)

df_knn[knn_columns] = knn_imputer.fit_transform(
    df_knn[knn_columns]
)

print("Missing Values After KNN:")
print(df_knn[knn_columns].isnull().sum())

Missing Values After KNN:
transaction_amount    0
age                   0
annual_income         0
dtype: int64


**Q6(f) MICE Algorithm**

In [116]:

df_mice = merged_df.copy()

mice_columns = df_mice.select_dtypes(
    include=["int64", "float64"]
).columns

mice_imputer = IterativeImputer(
    max_iter=10,
    random_state=42
)

df_mice[mice_columns] = mice_imputer.fit_transform(
    df_mice[mice_columns]
)

print("Missing Values After MICE:")
print(df_mice[mice_columns].isnull().sum())

Missing Values After MICE:
transaction_amount    0
age                   0
annual_income         0
dtype: int64


**Q6(g) Complete Case Analysis**

In [122]:
df_complete = df.dropna()

print("Original Number of Rows:")
print(len(merged_df))

print("\nRows After Complete Case Analysis:")
print(len(df_complete))

print("\nRows Removed:")
print(len(merged_df) - len(df_complete))

Original Number of Rows:
1500

Rows After Complete Case Analysis:
2086

Rows Removed:
-586


## PART D — Outlier Handling

**Q7(a) — Z-Score Method**

In [123]:

df_zscore = df.copy()

# Missing values ko median se fill
df_zscore["annual_income"] = df_zscore["annual_income"].fillna(
    df_zscore["annual_income"].median()
)

# Calculate Z-score
df_zscore["income_zscore"] = zscore(
    df_zscore["annual_income"]
)

# Detect outliers
zscore_outliers = df_zscore[
    df_zscore["income_zscore"].abs() > 3
]

print("========== Z-SCORE METHOD ==========")
print("Number of Outliers:", len(zscore_outliers))

print("\nOutlier Records:")
print(zscore_outliers[
    ["customer_id", "annual_income", "income_zscore"]
].head())

========== Z-SCORE METHOD ==========
Number of Outliers: 41

Outlier Records:
    customer_id  annual_income  income_zscore
10        C0021     1753660.63       3.187257
109       C0021     1753660.63       3.187257
161       C0044     1834892.56       3.433688
162       C0044     1834892.56       3.433688
163       C0044     1834892.56       3.433688


**Q7(b) — IQR Method**

In [119]:
df_iqr = df.copy()

# Fill missing values
df_iqr["loan_amount"] = df_iqr["loan_amount"].fillna(
    df_iqr["loan_amount"].median()
)

Q1 = df_iqr["loan_amount"].quantile(0.25)
Q3 = df_iqr["loan_amount"].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

print("========== IQR METHOD ==========")
print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Limit:", lower_limit)
print("Upper Limit:", upper_limit)

========== IQR METHOD ==========
Q1: 204275.64
Q3: 427126.29
IQR: 222850.64999999997
Lower Limit: -130000.33499999996
Upper Limit: 761402.2649999999


**Q7(c) — Percentile Method**

In [124]:
df_percentile = df.copy()

df_percentile["annual_income"] = df_percentile[
    "annual_income"
].fillna(
    df_percentile["annual_income"].median()
)

lower_percentile = df_percentile[
    "annual_income"
].quantile(0.01)

upper_percentile = df_percentile[
    "annual_income"
].quantile(0.99)

print("========== PERCENTILE METHOD ==========")
print("1% Percentile:", lower_percentile)
print("99% Percentile:", upper_percentile)

========== PERCENTILE METHOD ==========
1% Percentile: 241497.1582
99% Percentile: 1834892.56


**Q7(d) — Winsorization**

In [128]:
numeric_columns = df.select_dtypes(include="number").columns.tolist()

print("Numerical Columns:")
print(numeric_columns)

# First numerical column select
column = numeric_columns[0]

df_winsor = df.copy()

# Missing values handle
df_winsor[column] = df_winsor[column].fillna(
    df_winsor[column].median()
)

# Winsorization
df_winsor[column + "_winsorized"] = winsorize(
    df_winsor[column].to_numpy(),
    limits=[0.05, 0.05]
)

print("========== WINSORIZATION ==========")

print(
    df_winsor[
        [column, column + "_winsorized"]
    ].head(10)
)

Numerical Columns:
['transaction_amount', 'age', 'annual_income', 'loan_amount', 'emi_amount', 'days_late', 'year', 'inflation', 'gdp_growth', 'unemployment']
========== WINSORIZATION ==========
   transaction_amount  transaction_amount_winsorized
0             7625.32                        7625.32
1             9527.03                        9527.03
2             1243.62                        1244.60
3             2444.64                        2444.64
4             2444.64                        2444.64
5             2444.64                        2444.64
6             2432.12                        2432.12
7             2432.12                        2432.12
8             2121.57                        2121.57
9             8737.79                        8737.79


## PART E — Feature Engineering

**Q8(a) Mixed Variables**

In [ ]:
df_mixed = df.copy()

numeric_columns = df_mixed.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = df_mixed.select_dtypes(
    include=["object"]
).columns.tolist()

print("========== MIXED VARIABLES ==========")

print("\nNumerical Variables:")
print(numeric_columns)

print("\nCategorical Variables:")
print(categorical_columns)

========== MIXED VARIABLES ==========

Numerical Variables:
['transaction_amount', 'age', 'annual_income', 'loan_amount', 'emi_amount', 'days_late']

Categorical Variables:
['transaction_id', 'customer_id', 'transaction_date', 'transaction_type', 'payment_method', 'merchant_category', 'gender', 'city', 'occupation', 'education_level', 'loan_id', 'repayment_status', 'repayment_date']


C:\Users\Dell\AppData\Local\Temp\ipykernel_28564\254659231.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df_mixed.select_dtypes(


**Q8(b) Date & Time Variables**

In [129]:
df_date = df.copy()

# Convert transaction_date into datetime
df_date["transaction_date"] = pd.to_datetime(
    df_date["transaction_date"],
    errors="coerce"
)

# Extract Year
df_date["transaction_year"] = df_date[
    "transaction_date"
].dt.year

# Extract Month
df_date["transaction_month"] = df_date[
    "transaction_date"
].dt.month

# Extract Day
df_date["transaction_day"] = df_date[
    "transaction_date"
].dt.day

# Extract Weekday
df_date["transaction_weekday"] = df_date[
    "transaction_date"
].dt.day_name()

print("========== DATE FEATURES ==========")

print(
    df_date[
        [
            "transaction_date",
            "transaction_year",
            "transaction_month",
            "transaction_day",
            "transaction_weekday"
        ]
    ].head()
)

========== DATE FEATURES ==========
  transaction_date  transaction_year  transaction_month  transaction_day  \
0       2025-07-08              2025                  7                8   
1       2025-03-26              2025                  3               26   
2       2022-02-16              2022                  2               16   
3       2022-11-20              2022                 11               20   
4       2022-11-20              2022                 11               20   

  transaction_weekday  
0             Tuesday  
1           Wednesday  
2           Wednesday  
3              Sunday  
4              Sunday  


## Q9 — Encoding Categorical Variables

**Q9(a) Ordinal Encoding — education_level**

In [ ]:

df_ordinal = merged_df.copy()

education_order = [
    ["Primary", "Secondary", "Graduate", "Post-Graduate"]
]

ordinal_encoder = OrdinalEncoder(
    categories=education_order,
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

df_ordinal["education_level_encoded"] = (
    ordinal_encoder.fit_transform(
        df_ordinal[["education_level"]]
    )
)

print("========== ORDINAL ENCODING ==========")

print(
    df_ordinal[
        [
            "education_level",
            "education_level_encoded"
        ]
    ].head(10)
)

========== ORDINAL ENCODING ==========
  education_level  education_level_encoded
0   Post-Graduate                      3.0
1        Graduate                      2.0
2       Secondary                      1.0
3   Post-Graduate                      3.0
4        Graduate                      2.0
5   Post-Graduate                      3.0
6       Secondary                      1.0
7       Secondary                      1.0
8   Post-Graduate                      3.0
9       Secondary                      1.0


**Q9(b) Label Encoding — Binary Feature**

In [ ]:

df_label = merged_df.copy()

label_encoder = LabelEncoder()

df_label["gender_encoded"] = label_encoder.fit_transform(
    df_label["gender"].astype(str)
)

print("========== LABEL ENCODING ==========")

print(
    df_label[
        [
            "gender",
            "gender_encoded"
        ]
    ].head(10)
)

========== LABEL ENCODING ==========
   gender  gender_encoded
0    Male               1
1    Male               1
2  Female               0
3  Female               0
4    Male               1
5    Male               1
6  Female               0
7  Female               0
8    Male               1
9  Female               0


**Q9(c) One-Hot Encoding — Region & Loan Purpose**

In [130]:
df_onehot = df.copy()

# Find categorical columns automatically
categorical_columns = df_onehot.select_dtypes(
    include="object"
).columns.tolist()

print("Categorical Columns:")
print(categorical_columns)

# One-Hot Encoding
df_onehot = pd.get_dummies(
    df_onehot,
    columns=categorical_columns,
    dtype=int
)

print("\n========== ONE-HOT ENCODING ==========")

print(df_onehot.head())

print("\nNew Columns:")
print(df_onehot.columns.tolist())

Categorical Columns:
['transaction_id', 'customer_id', 'transaction_type', 'payment_method', 'merchant_category', 'gender', 'city', 'occupation', 'education_level', 'loan_id', 'repayment_status', 'repayment_date']

========== ONE-HOT ENCODING ==========
  transaction_date  transaction_amount  age  annual_income  loan_amount  \
0       2025-07-08             7625.32   59      746575.63          NaN   
1       2025-03-26             9527.03   46      562968.74          NaN   
2       2022-02-16             1243.62   47      518877.58    103499.95   
3       2022-11-20             2444.64   60     1499316.55    764195.94   
4       2022-11-20             2444.64   60     1499316.55   1055419.17   

   emi_amount  days_late  year  inflation  gdp_growth  ...  \
0         NaN        NaN  2025        4.6         6.8  ...   
1         NaN        NaN  2025        4.6         6.8  ...   
2    42666.00        0.0  2022        6.7         7.0  ...   
3    26162.78       60.0  2022        6.7      

C:\Users\Dell\AppData\Local\Temp\ipykernel_28564\123730727.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df_onehot.select_dtypes(


## Q10 — Encoding Numerical Features

**Q10(a) Binning — Discretize Income into Groups**

In [ ]:
df_binning = df.copy()

df_binning["income_group"] = pd.cut(
    df_binning["annual_income"],
    bins=3,
    labels=[
        "Low",
        "Medium",
        "High"
    ]
)

print("========== INCOME BINNING ==========")

print(
    df_binning[
        [
            "annual_income",
            "income_group"
        ]
    ].head(10)
)

========== INCOME BINNING ==========
   annual_income income_group
0      746575.63          Low
1      562968.74          Low
2      518877.58          Low
3     1499316.55       Medium
4     1499316.55       Medium
5     1499316.55       Medium
6      420889.44          Low
7      420889.44          Low
8      629948.02          Low
9      911038.82          Low


**Q10(b) Binarization — Credit Score > 700**

In [131]:
df_binary = df.copy()

# Find numerical columns
numeric_columns = df_binary.select_dtypes(
    include="number"
).columns.tolist()

print("Numerical Columns:")
print(numeric_columns)

# Select first numerical column
column = numeric_columns[0]

# Median se missing values fill
df_binary[column] = df_binary[column].fillna(
    df_binary[column].median()
)

# Binarization using median as threshold
threshold = df_binary[column].median()

df_binary[column + "_binary"] = (
    df_binary[column] >= threshold
).astype(int)

print("\n========== BINARIZATION ==========")

print("Selected Column:", column)
print("Threshold:", threshold)

print(
    df_binary[
        [
            column,
            column + "_binary"
        ]
    ].head(10)
)

Numerical Columns:
['transaction_amount', 'age', 'annual_income', 'loan_amount', 'emi_amount', 'days_late', 'year', 'inflation', 'gdp_growth', 'unemployment']

========== BINARIZATION ==========
Selected Column: transaction_amount
Threshold: 4485.625
   transaction_amount  transaction_amount_binary
0             7625.32                          1
1             9527.03                          1
2             1243.62                          0
3             2444.64                          0
4             2444.64                          0
5             2444.64                          0
6             2432.12                          0
7             2432.12                          0
8             2121.57                          0
9             8737.79                          1


**Q10(c) Quantile Binning**

In [132]:
df_quantile = df.copy()

# Find numerical columns
numeric_columns = df_quantile.select_dtypes(
    include="number"
).columns.tolist()

print("Numerical Columns:")
print(numeric_columns)

# Select first numerical column
column = numeric_columns[0]

# Missing values handle
df_quantile[column] = df_quantile[column].fillna(
    df_quantile[column].median()
)

# Quantile Binning
df_quantile["quantile_group"] = pd.qcut(
    df_quantile[column],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"],
    duplicates="drop"
)

print("\n========== QUANTILE BINNING ==========")

print("Selected Column:", column)

print(
    df_quantile[
        [
            column,
            "quantile_group"
        ]
    ].head(10)
)

Numerical Columns:
['transaction_amount', 'age', 'annual_income', 'loan_amount', 'emi_amount', 'days_late', 'year', 'inflation', 'gdp_growth', 'unemployment']

========== QUANTILE BINNING ==========
Selected Column: transaction_amount
   transaction_amount quantile_group
0             7625.32             Q3
1             9527.03             Q4
2             1243.62             Q1
3             2444.64             Q1
4             2444.64             Q1
5             2444.64             Q1
6             2432.12             Q1
7             2432.12             Q1
8             2121.57             Q1
9             8737.79             Q4


**Q10(d) K-Means Binning**

In [ ]:
from sklearn.cluster import KMeans

df_kmeans = merged_df.copy()

# Missing values fill
income_data = df_kmeans[
    ["annual_income"]
].fillna(
    df_kmeans["annual_income"].median()
)

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

df_kmeans["income_kmeans_cluster"] = kmeans.fit_predict(
    income_data
)

print("========== K-MEANS BINNING ==========")

print(
    df_kmeans[
        [
            "annual_income",
            "income_kmeans_cluster"
        ]
    ].head(10)
)

========== K-MEANS BINNING ==========
   annual_income  income_kmeans_cluster
0      746575.63                      0
1      562968.74                      2
2      518877.58                      2
3     1499316.55                      1
4      420889.44                      2
5      629948.02                      0
6      911038.82                      0
7     1753660.63                      1
8      746575.63                      0
9      914560.25                      0


## Part F — Feature Scaling

**Q11(a) Standardization — Z-Score Scaling**

In [133]:
from sklearn.preprocessing import StandardScaler

df_standard = df.copy()

# Find available numerical columns
scaling_columns = df_standard.select_dtypes(
    include="number"
).columns.tolist()

print("Numerical Columns Used:")
print(scaling_columns)

# Missing values fill with median
df_standard[scaling_columns] = df_standard[
    scaling_columns
].fillna(
    df_standard[scaling_columns].median()
)

# Standardization
scaler = StandardScaler()

df_standard[
    [col + "_standard" for col in scaling_columns]
] = scaler.fit_transform(
    df_standard[scaling_columns]
)

print("\n========== STANDARDIZATION ==========")

print(
    df_standard[
        [col + "_standard" for col in scaling_columns]
    ].head()
)

Numerical Columns Used:
['transaction_amount', 'age', 'annual_income', 'loan_amount', 'emi_amount', 'days_late', 'year', 'inflation', 'gdp_growth', 'unemployment']

========== STANDARDIZATION ==========
   transaction_amount_standard  age_standard  annual_income_standard  \
0                     0.246448      1.166576                0.132097   
1                     0.570814      0.140534               -0.424905   
2                    -0.842050      0.219460               -0.558663   
3                    -0.637197      1.245502                2.415662   
4                    -0.637197      1.245502                2.415662   

   loan_amount_standard  emi_amount_standard  days_late_standard  \
0             -0.247104            -0.006131           -0.409888   
1             -0.247104            -0.006131           -0.409888   
2             -1.098992             1.196203           -0.772569   
3              1.873654            -0.189105            1.403517   
4              3.183945 

**Q11(b) Normalization**

In [134]:
from sklearn.preprocessing import Normalizer

df_normalized = df.copy()

# Numerical columns automatically select
scaling_columns = df_normalized.select_dtypes(
    include="number"
).columns.tolist()

# Year ko normalization se remove
if "year" in scaling_columns:
    scaling_columns.remove("year")

print("Columns Used for Normalization:")
print(scaling_columns)

# Missing values fill with median
df_normalized[scaling_columns] = df_normalized[
    scaling_columns
].fillna(
    df_normalized[scaling_columns].median()
)

# Normalization
normalizer = Normalizer()

df_normalized[
    [col + "_normalized" for col in scaling_columns]
] = normalizer.fit_transform(
    df_normalized[scaling_columns]
)

print("\n========== NORMALIZATION ==========")

print(
    df_normalized[
        [col + "_normalized" for col in scaling_columns]
    ].head()
)

Columns Used for Normalization:
['transaction_amount', 'age', 'annual_income', 'loan_amount', 'emi_amount', 'days_late', 'inflation', 'gdp_growth', 'unemployment']

========== NORMALIZATION ==========
   transaction_amount_normalized  age_normalized  annual_income_normalized  \
0                       0.009502        0.000074                  0.930323   
1                       0.014997        0.000072                  0.886172   
2                       0.002343        0.000089                  0.977505   
3                       0.001453        0.000036                  0.890836   
4                       0.001333        0.000033                  0.817618   

   loan_amount_normalized  emi_amount_normalized  days_late_normalized  \
0                0.364913               0.035318              0.000012   
1                0.460960               0.044614              0.000016   
2                0.194982               0.080378              0.000000   
3                0.454056         

**Q11(c) Min-Max Scaling**

In [135]:
from sklearn.preprocessing import MinMaxScaler

df_minmax = df.copy()

# Numerical columns automatically select
scaling_columns = df_minmax.select_dtypes(
    include="number"
).columns.tolist()

# Year ko scaling se remove
if "year" in scaling_columns:
    scaling_columns.remove("year")

print("Columns Used for Min-Max Scaling:")
print(scaling_columns)

# Missing values fill with median
df_minmax[scaling_columns] = df_minmax[
    scaling_columns
].fillna(
    df_minmax[scaling_columns].median()
)

# Min-Max Scaling
minmax_scaler = MinMaxScaler()

df_minmax[
    [col + "_minmax" for col in scaling_columns]
] = minmax_scaler.fit_transform(
    df_minmax[scaling_columns]
)

print("\n========== MIN-MAX SCALING ==========")

print(
    df_minmax[
        [col + "_minmax" for col in scaling_columns]
    ].head()
)

Columns Used for Min-Max Scaling:
['transaction_amount', 'age', 'annual_income', 'loan_amount', 'emi_amount', 'days_late', 'inflation', 'gdp_growth', 'unemployment']

========== MIN-MAX SCALING ==========
   transaction_amount_minmax  age_minmax  annual_income_minmax  \
0                   0.069678    0.863636              0.251194   
1                   0.088022    0.568182              0.164589   
2                   0.008119    0.590909              0.143791   
3                   0.019704    0.886364              0.606253   
4                   0.019704    0.886364              0.606253   

   loan_amount_minmax  emi_amount_minmax  days_late_minmax  inflation_minmax  \
0            0.111659           0.519529          0.111111               0.0   
1            0.111659           0.519529          0.111111               0.0   
2            0.025396           0.838437          0.000000               1.0   
3            0.326409           0.470997          0.666667               1.0  

**Q11(d) MaxAbs Scaling**

In [136]:
from sklearn.preprocessing import MaxAbsScaler

df_maxabs = df.copy()

# Numerical columns automatically select
scaling_columns = df_maxabs.select_dtypes(
    include="number"
).columns.tolist()

# Year ko scaling se remove
if "year" in scaling_columns:
    scaling_columns.remove("year")

print("Columns Used for MaxAbs Scaling:")
print(scaling_columns)

# Missing values fill with median
df_maxabs[scaling_columns] = df_maxabs[
    scaling_columns
].fillna(
    df_maxabs[scaling_columns].median()
)

# MaxAbs Scaling
maxabs_scaler = MaxAbsScaler()

df_maxabs[
    [col + "_maxabs" for col in scaling_columns]
] = maxabs_scaler.fit_transform(
    df_maxabs[scaling_columns]
)

print("\n========== MAXABS SCALING ==========")

print(
    df_maxabs[
        [col + "_maxabs" for col in scaling_columns]
    ].head()
)

Columns Used for MaxAbs Scaling:
['transaction_amount', 'age', 'annual_income', 'loan_amount', 'emi_amount', 'days_late', 'inflation', 'gdp_growth', 'unemployment']

========== MAXABS SCALING ==========
   transaction_amount_maxabs  age_maxabs  annual_income_maxabs  \
0                   0.073271    0.907692              0.319859   
1                   0.091544    0.707692              0.241196   
2                   0.011950    0.723077              0.222305   
3                   0.023490    0.923077              0.642360   
4                   0.023490    0.923077              0.642360   

   loan_amount_maxabs  emi_amount_maxabs  days_late_maxabs  inflation_maxabs  \
0            0.130576           0.567731          0.111111          0.686567   
1            0.130576           0.567731          0.111111          0.686567   
2            0.046150           0.854645          0.000000          1.000000   
3            0.340753           0.524068          0.666667          1.000000   


**Q11(e) Robust Scaling**

In [137]:
from sklearn.preprocessing import RobustScaler

df_robust = df.copy()

# Numerical columns automatically select
scaling_columns = df_robust.select_dtypes(
    include="number"
).columns.tolist()

# Year ko scaling se remove
if "year" in scaling_columns:
    scaling_columns.remove("year")

print("Columns Used for Robust Scaling:")
print(scaling_columns)

# Missing values fill with median
df_robust[scaling_columns] = df_robust[
    scaling_columns
].fillna(
    df_robust[scaling_columns].median()
)

# Robust Scaling
robust_scaler = RobustScaler()

df_robust[
    [col + "_robust" for col in scaling_columns]
] = robust_scaler.fit_transform(
    df_robust[scaling_columns]
)

print("\n========== ROBUST SCALING ==========")

print(
    df_robust[
        [col + "_robust" for col in scaling_columns]
    ].head()
)

Columns Used for Robust Scaling:
['transaction_amount', 'age', 'annual_income', 'loan_amount', 'emi_amount', 'days_late', 'inflation', 'gdp_growth', 'unemployment']

========== ROBUST SCALING ==========
   transaction_amount_robust  age_robust  annual_income_robust  \
0                   0.619793    0.636364              0.297149   
1                   0.995201    0.045455             -0.183493   
2                  -0.639990    0.090909             -0.298914   
3                  -0.402902    0.681818              2.267660   
4                  -0.402902    0.681818              2.267660   

   loan_amount_robust  emi_amount_robust  days_late_robust  inflation_robust  \
0            0.000000           0.000000          0.000000         -0.333333   
1            0.000000           0.000000          0.000000         -0.333333   
2           -0.849625           0.811353         -0.333333          1.666667   
3            2.115123          -0.123474          1.666667          1.666667   


## PART G — Feature Construction & Transformation

**Q12(a) FunctionTransformer**

In [139]:
from sklearn.preprocessing import FunctionTransformer
import numpy as np

df_function = df.copy()

# Find numerical columns
numeric_columns = df_function.select_dtypes(
    include="number"
).columns.tolist()

# Remove year if present
if "year" in numeric_columns:
    numeric_columns.remove("year")

# Select first numerical column
column = numeric_columns[0]

print("Column Used:", column)

# Missing values
df_function[column] = df_function[column].fillna(
    df_function[column].median()
)

# Make values positive
df_function[column + "_positive"] = (
    df_function[column]
    - df_function[column].min()
    + 1
)

# Log Transformation
log_transformer = FunctionTransformer(np.log1p)

df_function[column + "_log"] = (
    log_transformer.transform(
        df_function[[column + "_positive"]]
    ).iloc[:, 0]
)

# Reciprocal Transformation
df_function[column + "_reciprocal"] = (
    1 / df_function[column + "_positive"]
)

# Square Root Transformation
df_function[column + "_sqrt"] = np.sqrt(
    df_function[column + "_positive"]
)

print("\n========== FUNCTION TRANSFORMER ==========")

print(
    df_function[
        [
            column,
            column + "_log",
            column + "_reciprocal",
            column + "_sqrt"
        ]
    ].head(10)
)

Column Used: transaction_amount

========== FUNCTION TRANSFORMER ==========
   transaction_amount  transaction_amount_log  transaction_amount_reciprocal  \
0             7625.32                8.885359                       0.000138   
1             9527.03                9.119005                       0.000110   
2             1243.62                6.737809                       0.001187   
3             2444.64                7.623021                       0.000489   
4             2444.64                7.623021                       0.000489   
5             2444.64                7.623021                       0.000489   
6             2432.12                7.616879                       0.000492   
7             2432.12                7.616879                       0.000492   
8             2121.57                7.451044                       0.000581   
9             8737.79                9.028564                       0.000120   

   transaction_amount_sqrt  
0             

**Q12(b) PowerTransformer — Box-Cox**

In [ ]:
from sklearn.preprocessing import PowerTransformer

df_boxcox = df.copy()

df_boxcox["annual_income"] = df_boxcox[
    "annual_income"
].fillna(
    df_boxcox["annual_income"].median()
)

# Ensure positive values
df_boxcox["annual_income_positive"] = (
    df_boxcox["annual_income"] -
    df_boxcox["annual_income"].min() + 1
)

boxcox_transformer = PowerTransformer(
    method="box-cox"
)

df_boxcox["annual_income_boxcox"] = (
    boxcox_transformer.fit_transform(
        df_boxcox[["annual_income_positive"]]
    ).ravel()
)

print("========== BOX-COX TRANSFORMATION ==========")

print(
    df_boxcox[
        [
            "annual_income",
            "annual_income_boxcox"
        ]
    ].head()
)

========== BOX-COX TRANSFORMATION ==========
   annual_income  annual_income_boxcox
0      746575.63              0.338535
1      562968.74             -0.264762
2      518877.58             -0.438007
3     1499316.55              1.954339
4     1499316.55              1.954339


**Q12(c) PowerTransformer — Yeo-Johnson**

In [ ]:
df_yeojohnson = df.copy()

df_yeojohnson["loan_amount"] = (
    df_yeojohnson["loan_amount"].fillna(
        df_yeojohnson["loan_amount"].median()
    )
)

yeo_transformer = PowerTransformer(
    method="yeo-johnson"
)

df_yeojohnson["loan_amount_yeojohnson"] = (
    yeo_transformer.fit_transform(
        df_yeojohnson[["loan_amount"]]
    ).ravel()
)

print("========== YEO-JOHNSON TRANSFORMATION ==========")

print(
    df_yeojohnson[
        [
            "loan_amount",
            "loan_amount_yeojohnson"
        ]
    ].head()
)

========== YEO-JOHNSON TRANSFORMATION ==========
   loan_amount  loan_amount_yeojohnson
0    292839.41               -0.005012
1    292839.41               -0.005012
2    103499.95               -1.723912
3    764195.94                1.635850
4   1055419.17                2.200490


**Q12(d) ColumnTransformer**

In [140]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder

df_ct = df.copy()

# ==========================================
# Find Numerical Columns
# ==========================================

numeric_features = df_ct.select_dtypes(
    include="number"
).columns.tolist()

# Remove year if present
if "year" in numeric_features:
    numeric_features.remove("year")


# ==========================================
# Find Categorical Columns
# ==========================================

categorical_features = df_ct.select_dtypes(
    include="object"
).columns.tolist()

# Remove date column if present
if "transaction_date" in categorical_features:
    categorical_features.remove("transaction_date")


print("Numerical Features:")
print(numeric_features)

print("\nCategorical Features:")
print(categorical_features)


# ==========================================
# Fill Missing Values
# ==========================================

if numeric_features:
    df_ct[numeric_features] = df_ct[
        numeric_features
    ].fillna(
        df_ct[numeric_features].median()
    )

if categorical_features:
    df_ct[categorical_features] = df_ct[
        categorical_features
    ].fillna("Unknown")


# ==========================================
# Column Transformer
# ==========================================

transformers = []

# First numerical column → StandardScaler
if len(numeric_features) > 0:
    transformers.append(
        (
            "standard",
            StandardScaler(),
            [numeric_features[0]]
        )
    )

# Second numerical column → MinMaxScaler
if len(numeric_features) > 1:
    transformers.append(
        (
            "minmax",
            MinMaxScaler(),
            [numeric_features[1]]
        )
    )

# Remaining numerical columns → StandardScaler
if len(numeric_features) > 2:
    transformers.append(
        (
            "numeric_remaining",
            StandardScaler(),
            numeric_features[2:]
        )
    )

# Categorical columns → OneHotEncoder
if categorical_features:
    transformers.append(
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    )


# Create ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop"
)


# ==========================================
# Transform Data
# ==========================================

transformed_data = preprocessor.fit_transform(
    df_ct
)


print("\n========== COLUMN TRANSFORMER ==========")

print("Original Shape:", df_ct.shape)

print("Transformed Shape:", transformed_data.shape)

print("\nColumnTransformer completed successfully.")

Numerical Features:
['transaction_amount', 'age', 'annual_income', 'loan_amount', 'emi_amount', 'days_late', 'inflation', 'gdp_growth', 'unemployment']

Categorical Features:
['transaction_id', 'customer_id', 'transaction_type', 'payment_method', 'merchant_category', 'gender', 'city', 'occupation', 'education_level', 'loan_id', 'repayment_status', 'repayment_date']

========== COLUMN TRANSFORMER ==========
Original Shape: (2470, 23)
Transformed Shape: (2470, 3177)

ColumnTransformer completed successfully.


C:\Users\Dell\AppData\Local\Temp\ipykernel_28564\656939409.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = df_ct.select_dtypes(


**Q13(a) Debt-to-Income Ratio**

In [ ]:
df_features = df.copy()

df_features["annual_income"] = df_features[
    "annual_income"
].fillna(
    df_features["annual_income"].median()
)

df_features["loan_amount"] = df_features[
    "loan_amount"
].fillna(
    df_features["loan_amount"].median()
)

df_features["debt_to_income_ratio"] = (
    df_features["loan_amount"] /
    df_features["annual_income"]
)

print("========== DEBT-TO-INCOME RATIO ==========")

print(
    df_features[
        [
            "annual_income",
            "loan_amount",
            "debt_to_income_ratio"
        ]
    ].head(10)
)

========== DEBT-TO-INCOME RATIO ==========
   annual_income  loan_amount  debt_to_income_ratio
0      746575.63    292839.41              0.392243
1      562968.74    292839.41              0.520170
2      518877.58    103499.95              0.199469
3     1499316.55    764195.94              0.509696
4     1499316.55   1055419.17              0.703934
5     1499316.55    245235.32              0.163565
6      420889.44    317395.81              0.754107
7      420889.44    160662.91              0.381722
8      629948.02    663952.94              1.053981
9      911038.82    405521.17              0.445120


**Q13(b) Average Monthly Transactions**

In [141]:
df_features = df.copy()

# Numerical columns
numeric_columns = df_features.select_dtypes(
    include="number"
).columns.tolist()

# year ko remove
if "year" in numeric_columns:
    numeric_columns.remove("year")

# First numerical column
column = numeric_columns[0]

# Missing values
df_features[column] = df_features[column].fillna(
    df_features[column].median()
)

# Average monthly value
df_features["average_monthly_" + column] = (
    df_features[column] / 6
)

print("========== AVERAGE MONTHLY VALUE ==========")

print("Column Used:", column)

print(
    df_features[
        [
            column,
            "average_monthly_" + column
        ]
    ].head(10)
)

========== AVERAGE MONTHLY VALUE ==========
Column Used: transaction_amount
   transaction_amount  average_monthly_transaction_amount
0             7625.32                         1270.886667
1             9527.03                         1587.838333
2             1243.62                          207.270000
3             2444.64                          407.440000
4             2444.64                          407.440000
5             2444.64                          407.440000
6             2432.12                          405.353333
7             2432.12                          405.353333
8             2121.57                          353.595000
9             8737.79                         1456.298333
